In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
import joblib
import os

# Connect to workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

# Reload data and model from previous session
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Data and model ready")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Found the config file in: /config.json


Connected: ml-learning-workspace
Data and model ready
Training set size: 455
Test set size: 114


In [2]:
# Install evidently
import subprocess
subprocess.run(['pip', 'install', 'evidently', '--quiet'])
print("Evidently installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azure-cli 2.81.0 requires azure-datalake-store~=1.0.1, but you have azure-datalake-store 0.0.53 which is incompatible.
azure-cli 2.81.0 requires azure-keyvault-keys==4.11.0, but you have azure-keyvault-keys 4.8.0 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-keyvault==12.1.0, but you have azure-mgmt-keyvault 10.3.1 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-storage==24.0.0, but you have azure-mgmt-storage 22.0.0 which is incompatible.
azure-cli 2.81.0 requires websocket-client~=1.3.1, but you have websocket-client 1.9.0 which is incompatible.
azureml-automl-runtime 1.61.0 requires psutil<5.9.4,>=5.2.2, but you have psutil 7.2.2 which is incompatible.
azureml-automl-runtime 1.61.0 requires statsmodels<0.14,>=0.13.0, but you have statsmodels 0.14.6 which is incompatible.
azureml-core 1

Evidently installed


In [3]:
import pandas as pd
import numpy as np

# Reference data = training data (what model was trained on)
reference_data = X_train.copy()

# Simulate drifted current data
# In real life this would be new data coming in months later
# We simulate drift by shifting some feature distributions
np.random.seed(42)
current_data = X_test.copy()

# Introduce drift in top 3 most important features
# Simulate a population shift - patients are coming in with
# higher values for these features (e.g. different hospital population)
current_data['worst concave points'] = current_data['worst concave points'] * 1.8
current_data['mean concave points'] = current_data['mean concave points'] * 1.6
current_data['worst radius'] = current_data['worst radius'] + 5.0

print("Reference data (training) stats:")
print(reference_data[['worst concave points', 'mean concave points', 'worst radius']].describe().round(3))
print("\nCurrent data (drifted) stats:")
print(current_data[['worst concave points', 'mean concave points', 'worst radius']].describe().round(3))

Reference data (training) stats:
       worst concave points  mean concave points  worst radius
count               455.000              455.000       455.000
mean                  0.114                0.048        16.235
std                   0.065                0.038         4.811
min                   0.000                0.000         8.678
25%                   0.064                0.020        13.055
50%                   0.099                0.033        14.970
75%                   0.161                0.074        18.410
max                   0.291                0.201        36.040

Current data (drifted) stats:
       worst concave points  mean concave points  worst radius
count               114.000              114.000       114.000
mean                  0.209                0.082        21.405
std                   0.122                0.067         4.939
min                   0.000                0.000        12.930
25%                   0.125                0.032      

In [7]:
import evidently
print(dir(evidently))

['BinaryClassification', 'ColumnType', 'DataDefinition', 'Dataset', 'LLMClassification', 'MulticlassClassification', 'Recsys', 'Regression', 'Report', 'Run', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_pydantic_compat', '_registry', '_version', 'compare', 'core', 'descriptors', 'errors', 'generators', 'guardrails', 'legacy', 'llm', 'metrics', 'presets', 'pydantic_utils', 'sdk', 'tests', 'ui', 'utils', 'version_info']


In [8]:
from evidently import Dataset, DataDefinition, Report
from evidently.presets import DataDriftPreset
import os

os.makedirs("drift_reports", exist_ok=True)

# Create datasets
definition = DataDefinition()
ref_dataset = Dataset.from_pandas(reference_data, data_definition=definition)
cur_dataset = Dataset.from_pandas(current_data, data_definition=definition)

# Create and run report
report = Report([DataDriftPreset()])
my_report = report.run(reference_data=ref_dataset, current_data=cur_dataset)

# Save report
my_report.save_html("drift_reports/drift_report.html")
print("Drift report saved")

Drift report saved


In [9]:
# Get drift summary
result = my_report.dict()
print("Drift Report Summary:")
print(result)

Drift Report Summary:
{'metrics': [{'id': '15e89f895b482f9b84ba7274ed18a106', 'metric_name': 'DriftedColumnsCount(drift_share=0.5)', 'config': {'type': 'evidently:metric_v2:DriftedColumnsCount', 'drift_share': 0.5}, 'value': {'count': 4.0, 'share': 0.13333333333333333}}, {'id': '948ecccde5650972e62b83207b32e184', 'metric_name': 'ValueDrift(column=mean radius,method=K-S p_value,threshold=0.05)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'mean radius', 'method': 'K-S p_value', 'threshold': 0.05}, 'value': 0.9988145213462264}, {'id': 'eeef763bf1a4819cef98f873bd2a2f3b', 'metric_name': 'ValueDrift(column=mean texture,method=K-S p_value,threshold=0.05)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'mean texture', 'method': 'K-S p_value', 'threshold': 0.05}, 'value': 0.15621637952535372}, {'id': 'a24a4686515cd25067e44f0678e8859f', 'metric_name': 'ValueDrift(column=mean perimeter,method=K-S p_value,threshold=0.05)', 'config': {'type': 'evidently:metric_v

In [10]:
# Parse drift results clearly
metrics = result['metrics']

print("=" * 60)
print("DATA DRIFT REPORT SUMMARY")
print("=" * 60)

# Overall drift
overall = metrics[0]
drifted_count = overall['value']['count']
drifted_share = overall['value']['share']
print(f"\nTotal features: 30")
print(f"Drifted features: {int(drifted_count)}")
print(f"Share drifted: {drifted_share:.1%}")
print(f"Drift detected: {'YES ⚠️' if drifted_count > 0 else 'NO ✅'}")

# Individual feature drift
print("\n" + "=" * 60)
print("FEATURE DRIFT DETAILS (p-value < 0.05 = drift detected)")
print("=" * 60)

drifted_features = []
stable_features = []

for metric in metrics[1:]:
    feature = metric['metric_name'].split('column=')[1].split(',')[0]
    p_value = metric['value']
    drifted = p_value < 0.05
    
    if drifted:
        drifted_features.append((feature, p_value))
    else:
        stable_features.append((feature, p_value))

print("\n🚨 DRIFTED FEATURES:")
for feature, p_value in drifted_features:
    print(f"  - {feature}: p-value = {p_value:.2e} ← DRIFT DETECTED")

print(f"\n✅ STABLE FEATURES: {len(stable_features)} features showing no significant drift")

DATA DRIFT REPORT SUMMARY

Total features: 30
Drifted features: 4
Share drifted: 13.3%
Drift detected: YES ⚠️

FEATURE DRIFT DETAILS (p-value < 0.05 = drift detected)

🚨 DRIFTED FEATURES:
  - mean smoothness: p-value = 4.65e-02 ← DRIFT DETECTED
  - mean concave points: p-value = 4.50e-06 ← DRIFT DETECTED
  - worst radius: p-value = 7.43e-27 ← DRIFT DETECTED
  - worst concave points: p-value = 1.29e-11 ← DRIFT DETECTED

✅ STABLE FEATURES: 26 features showing no significant drift
